# Lab 3: Data Wrangling

**Goal:** Take `dataset_part_1.csv` (from Lab 1) and turn the messy `Outcome` field into a clean binary `Class` label: **1 = the booster landed successfully, 0 = it did not.** This `Class` column is the target variable for the predictive model in Lab 8.

We'll also inspect missing values and summarize launch sites / orbits, since your report's Data Wrangling slide needs to explain the cleaning process.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Load the dataset from Lab 1
Upload `dataset_part_1.csv` to this Colab session first (folder icon on the left → upload), or read it straight from GitHub if you already pushed it there.

In [2]:
import os

if os.path.exists('dataset_part_1.csv'):
    df = pd.read_csv('dataset_part_1.csv')
else:
    # Not found locally in this session -- load the same dataset from IBM's hosted copy
    print('dataset_part_1.csv not found locally, loading from IBM dataset repository...')
    fallback_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv'
    df = pd.read_csv(fallback_url)

df.head(10)

dataset_part_1.csv not found locally, loading from IBM dataset repository...


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
5,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857
6,7,2014-04-18,Falcon 9,2296.000000,ISS,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1006,-80.577366,28.561857
7,8,2014-07-14,Falcon 9,1316.000000,LEO,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1007,-80.577366,28.561857
8,9,2014-08-05,Falcon 9,4535.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1008,-80.577366,28.561857
9,10,2014-09-07,Falcon 9,4428.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1011,-80.577366,28.561857


## 2. Identify missing values

In [3]:
df.isnull().sum() / df.shape[0] * 100

,0
FlightNumber,0.000000
Date,0.000000
BoosterVersion,0.000000
PayloadMass,0.000000
Orbit,0.000000
LaunchSite,0.000000
Outcome,0.000000
Flights,0.000000
GridFins,0.000000
Reused,0.000000


In [4]:
df.dtypes

,0
FlightNumber,int64
Date,object
BoosterVersion,object
PayloadMass,float64
Orbit,object
LaunchSite,object
Outcome,object
Flights,int64
GridFins,bool
Reused,bool


Note: `LandingPad` will have missing values by design — a missing landing pad ID simply means the booster wasn't attempting a ground/drone-ship landing at all (e.g. it was expended). We'll leave those as NaN rather than dropping rows, since that's informative in itself.

## 3. Summarize launch sites and orbits

In [5]:
print("Launches per site:")
print(df['LaunchSite'].value_counts())
print()
print("Launches per orbit:")
print(df['Orbit'].value_counts())

Launches per site:
LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

Launches per orbit:
Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
HEO       1
ES-L1     1
SO        1
GEO       1
Name: count, dtype: int64


## 4. Build the landing outcome label
The `Outcome` column looks like `'True ASDS'`, `'False ASDS'`, `'True RTLS'`, `'None None'`, etc. — a landing success/failure flag plus a landing type. We split this into a **bad outcomes** set (failures, or no landing attempted) and everything else counts as a successful landing.

In [6]:
print(df['Outcome'].value_counts())

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64


In [7]:
landing_outcomes = df['Outcome'].value_counts()

bad_outcomes = set(landing_outcomes.keys()[[1, 3, 5, 6, 7]])
print("Bad outcomes:", bad_outcomes)

Bad outcomes: {'False Ocean', 'False ASDS', 'None None', 'None ASDS', 'False RTLS'}


**Important:** the indices `[1, 3, 5, 6, 7]` above depend on the order `value_counts()` printed in the previous cell. Re-check that printout for *your* data — the bad outcomes should be things like `False ASDS`, `False Ocean`, `False RTLS`, `None ASDS`, `None None` (i.e. anything starting with `False`, plus attempts with no confirmed landing type). Adjust the index list if your ordering differs.

In [8]:
landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df['Outcome']]
df['Class'] = landing_class

df[['Outcome', 'Class']].head(10)

,Outcome,Class
0,None None,0
1,None None,0
2,None None,0
3,False Ocean,0
4,None None,0
5,None None,0
6,True Ocean,1
7,True Ocean,1
8,None None,0
9,None None,0


In [9]:
success_rate = df['Class'].mean()
print(f"Overall landing success rate: {success_rate:.2%}")

Overall landing success rate: 66.67%


## 5. Export the wrangled dataset for EDA and modeling

In [10]:
df.to_csv('dataset_part_2.csv', index=False)
print('Saved dataset_part_2.csv with', df.shape[0], 'rows and', df.shape[1], 'columns')

Saved dataset_part_2.csv with 90 rows and 18 columns
